In [10]:
# Load env variables and create client
import json
import logging
import os
import sys
import re
import ast

from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic(api_key=os.getenv("CLAUDE_COURSE_API_KEY"))
model = "claude-sonnet-4-0"

In [2]:
eval_data = [
    {"animal_statement": "The animal is a human.", "golden_answer": "2"},
    {"animal_statement": "The animal is a snake.", "golden_answer": "0"},
    {"animal_statement": "The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.", "golden_answer": "5"},
    {"animal_statement": "The animal is a dog.", "golden_answer": "4"},
    {"animal_statement": "The animal is a cat with two extra legs.", "golden_answer": "6"},
    {"animal_statement": "The animal is an elephant.", "golden_answer": "4"},
    {"animal_statement": "The animal is a bird.", "golden_answer": "2"},
    {"animal_statement": "The animal is a fish.", "golden_answer": "0"},
    {"animal_statement": "The animal is a spider with two extra legs", "golden_answer": "10"},
    {"animal_statement": "The animal is an octopus.", "golden_answer": "8"},
    {"animal_statement": "The animal is an octopus that lost two legs and then regrew three legs.", "golden_answer": "9"},
    {"animal_statement": "The animal is a two-headed, eight-legged mythical creature.", "golden_answer": "8"},
]

In [3]:
def build_input_prompt(animal_statement):
    user_content = f"""You will be provided a statement about an animal and your job is to determine how many legs that animal has.
    
    Here is the animal statement.
    <animal_statement>{animal_statement}</animal_statement>
    
    How many legs does the animal have? Please respond with a number"""

    messages = [{'role': 'user', 'content': user_content}]
    return messages

In [5]:
build_input_prompt(eval_data[1]['animal_statement'])

[{'role': 'user',
  'content': 'You will be provided a statement about an animal and your job is to determine how many legs that animal has.\n\n    Here is the animal statement.\n    <animal_statement>The animal is a snake.</animal_statement>\n\n    How many legs does the animal have? Please respond with a number'}]

In [11]:


def get_completion(messages):
    response = client.messages.create(
        model=model,
        max_tokens=200,
        messages=messages
    )
    return response.content[0].text

In [13]:
full_prompt = build_input_prompt(eval_data[1]['animal_statement'])
get_completion(full_prompt)

'0'

In [14]:
outputs = [get_completion(build_input_prompt(question['animal_statement'])) for question in eval_data]

In [15]:
outputs

['2',
 '0',
 'To solve this, I need to:\n1. Determine how many legs a fox normally has\n2. Apply the changes described in the statement\n\nA fox normally has 4 legs.\n\nAccording to the statement:\n- The fox lost a leg (4 - 1 = 3 legs)\n- Then magically grew back the leg he lost (3 + 1 = 4 legs)\n- And grew a mysterious extra leg on top of that (4 + 1 = 5 legs)\n\n5',
 '4',
 'A normal cat has 4 legs. If this cat has two extra legs, then it would have 4 + 2 = 6 legs.\n\n6',
 '4',
 '2',
 '0',
 'Looking at this step by step:\n\n- A regular spider has 8 legs\n- This spider has "two extra legs" \n- 8 + 2 = 10\n\n10',
 '0\n\nAn octopus has 8 arms, not legs. Arms and legs are different appendages - legs are used for walking/standing while arms are used for grasping and manipulation. Since the question specifically asks about legs, the answer is 0.',
 'An octopus normally has 8 legs.\n\nIf it lost 2 legs: 8 - 2 = 6 legs\nIf it then regrew 3 legs: 6 + 3 = 9 legs\n\n9',
 '8']

In [17]:
for output, question in zip(outputs, eval_data):
    print(f"Animal Statement: {question['animal_statement']}\nGolden Answer: {question['golden_answer']}\nOutput: {output}\n")

Animal Statement: The animal is a human.
Golden Answer: 2
Output: 2

Animal Statement: The animal is a snake.
Golden Answer: 0
Output: 0

Animal Statement: The fox lost a leg, but then magically grew back the leg he lost and a mysterious extra leg on top of that.
Golden Answer: 5
Output: To solve this, I need to:
1. Determine how many legs a fox normally has
2. Apply the changes described in the statement

A fox normally has 4 legs.

According to the statement:
- The fox lost a leg (4 - 1 = 3 legs)
- Then magically grew back the leg he lost (3 + 1 = 4 legs)
- And grew a mysterious extra leg on top of that (4 + 1 = 5 legs)

5

Animal Statement: The animal is a dog.
Golden Answer: 4
Output: 4

Animal Statement: The animal is a cat with two extra legs.
Golden Answer: 6
Output: A normal cat has 4 legs. If this cat has two extra legs, then it would have 4 + 2 = 6 legs.

6

Animal Statement: The animal is an elephant.
Golden Answer: 4
Output: 4

Animal Statement: The animal is a bird.
Golden

In [18]:
def grade_completion(output, golden_answer):
    return output == golden_answer

grades = [grade_completion(output, question['golden_answer']) for output, question in zip(outputs, eval_data)]
print(f"Score: {sum(grades)/len(grades)*100}%")

Score: 58.333333333333336%
